# End-to-End Pipeline Demo: Injection & Detection

This notebook demonstrates how to:
1. Load a Rubin `calexp` or `deepCoadd` image via Butler on the RSP.
2. Inject custom star cluster profiles.
3. Run the matched-filter multi-scale detection pipeline.
4. Compute the Multiple Concentration Index (MCI) to differentiate clusters from stars.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Ensure our pipeline is in the python path
pipeline_path = os.path.expanduser('~/INJECT/star-cluster-injection-pipeline')
if pipeline_path not in sys.path:
    sys.path.insert(0, pipeline_path)

# RSP Butler
from lsst.daf.butler import Butler

# Pipeline imports
from src.detection import run_cluster_detection, plot_detection_diagnostic, plot_mci_diagram
from src.light_profiles import PlummerProfile, KingProfile

plt.style.use('tableau-colorblind10')
print("Imports successful!")

## 1. Load Data with Butler

In [ ]:
b = Butler('dp02', collections='2.2i/runs/DP0.2')
dataId = {'tract': 3828, 'patch': 24, 'band': 'i'}
coadd = b.get('deepCoadd', dataId=dataId)

image_array = coadd.image.array.copy()
print(f"Loaded deepCoadd shape: {image_array.shape}")

## 2. Run Detection Pipeline on the Image
(In a real test, you'd inject sources here first using `inject_cluster`)

In [ ]:
# Run the full detection + MCI filtering
detections = run_cluster_detection(
    image=image_array[:1000, :1000],  # run on a cutout for speed in demo
    psf_fwhm=3.5,                     # approximate PSF in pixels
    threshold_sigma=5.0,
    use_multiscale=True,
    use_mci=True,
    mci_max=0.85                      # select extended objects
)

print(f"\nDetected {len(detections)} extended cluster candidates!")

## 3. Visualize Diagnostics

In [ ]:
# Plot the MCI diagram (extended sources vs point sources)
fig_mci = plot_mci_diagram(detections)
plt.show()